In [ ]:
# Library Imports

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import geom
from scipy.stats.sampling import DiscreteAliasUrn
from scipy.stats import chisquare
import time
import random
import math
import simpy
import pandas as pd
from collections import Counter
from scipy.stats import ks_1samp
from scipy.linalg import expm
from lifelines.statistics import logrank_test
from scipy.linalg import norm

### Part 3: Estimation
Task 12

In [ ]:
# Use the same Q matrix as defined in Task 7
Q = np.array([
    [-0.0085,  0.005,  0.0025,  0.000,  0.001],
    [ 0.0000, -0.014,  0.005,  0.004,  0.005],
    [ 0.0000,  0.000, -0.008,  0.003,  0.005],
    [ 0.0000,  0.000,  0.000, -0.009,  0.009],
    [ 0.0000,  0.000,  0.000,  0.000,  0.000]
])

# Re-derive P_embedded for the internal path generator
P_embedded = np.zeros_like(Q)
for i in range(4):
    total_exit_rate = -Q[i, i]
    P_embedded[i, :] = Q[i, :] / total_exit_rate
    P_embedded[i, i] = 0.0

P_embedded[4, 4] = 1.0 

# set the seed 
np.random.seed(42)
num_women = 1000
screening_interval = 48

observed_screening_series = []

for i in range(num_women):
    current_state = 0
    current_time = 0.0
    timeline = [(current_state, current_time)]
    
    while current_state != 4:
        rate = -Q[current_state, current_state]
        sojourn_time = np.random.exponential(scale=1.0 / rate)
        current_time += sojourn_time
        current_state = np.random.choice(5, p=P_embedded[current_state])
        timeline.append((current_state, current_time))
    
    # Extract the 4-Year Screenings Y(i)
    y_i = []
    check_time = 0.0
    patient_dead_and_recorded = False
    
    while not patient_dead_and_recorded:
        # Find what state the patient was in at exactly 'check_time'
        # We look for the last state entered before or exactly at check_time
        state_at_check = 0
        for state, time_entered in timeline:
            if time_entered <= check_time:
                state_at_check = state
            else:
                break
        
        # Convert index (0-4) to state labels (1-5) for proper project presentation
        y_i.append(state_at_check + 1)
        
        # If the state observed is 5 (Death), we stop tracking this patient
        if state_at_check == 4:
            patient_dead_and_recorded = True
            
        # Move to the next screening appointment (48 months later)
        check_time += screening_interval
        
    observed_screening_series.append(y_i)

print("--- Sample Screenings Vectors Y(i) (Observed State every 48 months) ---")
for idx in range(10):
    print(f"Woman {idx + 1}: {observed_screening_series[idx]}")


# The vectors will be of completely different lengths. 
# In 1st example, woman survived the first 48 months but died before month 96, thus her screening vecotr is [1, 1, 5].

Task 12: Visualisation of a sample

In [ ]:
# Plot of the Screening Vectors
sample_y = [
    [1, 1, 5],
    [1, 1, 1, 1, 5],
    [1, 1, 1, 5],
    [1, 1, 1, 1, 1, 3, 5],
    [1, 1, 2, 3, 4, 4, 5]
]

plt.figure(figsize=(10, 5))
state_labels = {1: 'Post-Op (1)', 2: 'Local (2)', 3: 'Distant (3)', 4: 'Joint (4)', 5: 'Dead (5)'}
colors = {1: '#1f77b4', 2: '#ff7f0e', 3: '#2ca02c', 4: '#d62728', 5: '#7f7f7f'}

# Plot each patient as a horizontal track
for idx, patient_vec in enumerate(sample_y):
    # X-axis points are screening intervals: 0, 48, 96, 144...
    times = [t * 48 for t in range(len(patient_vec))]
    
    # Draw the connecting baseline line
    plt.plot(times, [idx] * len(times), color='lightgray', zorder=1, linestyle='-')
    
    # Scatter plot the discrete screening observations
    for t_idx, state in enumerate(patient_vec):
        plt.scatter(times[t_idx], idx, color=colors[state], s=120, edgecolors='black', zorder=2)
        # Label the state next to the dot for clarity
        plt.text(times[t_idx], idx + 0.15, f"S{state}", ha='center', va='bottom', fontsize=9)

# Formatting
plt.yticks(range(len(sample_y)), [f"Patient {i+1}" for i in range(len(sample_y))])
plt.xlabel("Months Since Surgery (Screening Interval = 48 Months)", fontsize=11)
plt.suptitle("Visualizing Panel Data Observations Across 4-Year Screenings", fontsize=12)
plt.title("First 5 Simulatied Patient Observations", fontsize=12)
plt.xlim(-20, 350)
plt.ylim(-0.5, len(sample_y) - 0.2)
plt.grid(axis='x', linestyle=':', alpha=0.5)

# Custom legend
markers = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[s], markersize=10, markeredgecolor='black') for s in range(1, 6)]
plt.legend(markers, [state_labels[s] for s in range(1, 6)], loc='upper right', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

Task 13

In [ ]:
# Setup for simulation

# Extract all 48-month transition intervals from our Task 12 sequences
observed_pairs = []
for y_i in observed_screening_series:
    for k in range(len(y_i) - 1):
        # Convert from 1-5 state labels to 0-4 python indexing
        start_state = y_i[k] - 1
        end_state = y_i[k+1] - 1
        observed_pairs.append((start_state, end_state))

True_Q = Q.copy()
structure_mask = (Q > 0)

def sample_valid_segment(start_state, end_state, max_time, Q_current):
    # Compute the embedded discrete chain probabilities for the current Q guess
    P_embed = np.zeros_like(Q_current)
    for i in range(4):
        total_exit_rate = -Q_current[i, i]
        if total_exit_rate > 0:
            P_embed[i, :] = Q_current[i, :] / total_exit_rate
            P_embed[i, i] = 0.0
    P_embed[4, 4] = 1.0

    while True:
        current_state = start_state
        current_time = 0.0
        
        local_N = np.zeros_like(Q_current)
        local_S = np.zeros(5)
        
        # If already dead, they just stay dead for the whole 48 months
        if current_state == 4:
            if end_state == 4:
                local_S[4] += max_time
                return local_N, local_S
            else:
                continue 
        
        while current_time < max_time and current_state != 4:
            rate = -Q_current[current_state, current_state]
            sojourn = np.random.exponential(scale=1.0 / rate)
            
            # If the holding time goes past the 48-month check, cut it off
            if current_time + sojourn >= max_time:
                local_S[current_state] += (max_time - current_time)
                current_time = max_time
                break
            else:
                local_S[current_state] += sojourn
                current_time += sojourn
                next_state = np.random.choice(5, p=P_embed[current_state])
                local_N[current_state, next_state] += 1
                current_state = next_state
        
        # If the patient dies early, accumulate the remaining interval time in state 5
        if current_state == 4 and current_time < max_time:
            local_S[4] += (max_time - current_time)
            
        # Rejection check: Only return if the simulation ended in the correct state
        if current_state == end_state:
            return local_N, local_S


In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Initial guess Q(0)
Q_k = np.array([
    [-0.03,  0.01,  0.01,  0.00,  0.01],
    [ 0.00, -0.03,  0.01,  0.01,  0.01],
    [ 0.00,  0.00, -0.02,  0.01,  0.01],
    [ 0.00,  0.00,  0.00, -0.01,  0.01],
    [ 0.00,  0.00,  0.00,  0.00,  0.00]
])

max_iter = 10
tolerance = 1e-3
interval = 48.0

print("Running Monte Carlo EM iterations...")

# Main MCEM optimization loop
for iteration in range(max_iter):
    total_N = np.zeros_like(Q_k)
    total_S = np.zeros(5)
    
    # Loop over all patient panel observations (Steps 1 & 2)
    for y_i in observed_screening_series:
        for m in range(len(y_i) - 1):
            s_start = y_i[m] - 1
            s_end = y_i[m+1] - 1
            
            # Keep trying trajectories until we match the data boundaries
            seg_N, seg_S = sample_valid_segment(s_start, s_end, interval, Q_k)
            total_N += seg_N
            total_S += seg_S
            
    # Step 3: Standard MLE calculation (q_ij = N_ij / S_i)
    Q_next = np.zeros_like(Q_k)
    for i in range(4):
        for j in range(5):
            if i != j and structure_mask[i, j]:
                if total_S[i] > 0:
                    Q_next[i, j] = total_N[i, j] / total_S[i]
        
        # Force row sum constraint to zero, using Eq. 1 from the Project writeup
        Q_next[i, i] = -np.sum(Q_next[i, :])
        
    # Check convergence using the infinity norm
    diff = norm(Q_k - Q_next, np.inf)
    print(f"Iter {iteration+1:02d} - Diff: {diff:.6f}")
    
    Q_k = Q_next.copy()
    
    if diff < tolerance:
        print(f"Converged at iteration {iteration+1}!")
        break

# Display final comparison results
print("\nFinal Estimated Matrix Q:")
print(np.round(Q_k, 5))

print("\nTrue Generative Matrix Q:")
print(True_Q)